### Google Colab Setup

If running in Google Colab, run the cell below first to set up the environment.
If running locally (Jupyter), skip this cell.


In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Mount Drive and navigate to repo
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        # Generate data if needed
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        # Try cloned path
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')


# Case Study: Feature Selection on Store Data

Identify the most important features for predicting store revenue using multiple feature selection techniques.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.feature_selection import SelectKBest, f_regression, RFE, mutual_info_regression
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

sns.set_style('whitegrid')
%matplotlib inline
print('Libraries loaded!')


## 1. Load Store Data


In [ ]:
df = pd.read_csv('Data/store_data.csv')
print('Shape:', df.shape)
df.head()


In [ ]:
df.info()
print('\n---')
df.describe()


## 2. Baseline Model (all features)


In [ ]:
target = 'revenue'
X = df.drop(columns=['store_id', target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred = rf.predict(X_test_scaled)

print(f'R² Score (all features): {r2_score(y_test, y_pred):.4f}')
print(f'RMSE (all features): {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}')


## 3. Method 1: Feature Importance from Random Forest


In [ ]:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=importance, x='importance', y='feature')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

importance


## 4. Method 2: SelectKBest (ANOVA F-test)


In [ ]:
k = 5
selector_kbest = SelectKBest(score_func=f_regression, k=k)
selector_kbest.fit(X_train_scaled, y_train)

selected_kbest = X.columns[selector_kbest.get_support()]
print(f'Top {k} features by F-test:')
print(selected_kbest.tolist())

scores = pd.DataFrame({
    'feature': X.columns,
    'f_score': selector_kbest.scores_
}).sort_values('f_score', ascending=False)
scores


## 5. Method 3: Recursive Feature Elimination (RFE)


In [ ]:
rfe = RFE(estimator=LinearRegression(), n_features_to_select=5)
rfe.fit(X_train_scaled, y_train)

selected_rfe = X.columns[rfe.get_support()]
print('Features selected by RFE:')
print(selected_rfe.tolist())

rfe_ranking = pd.DataFrame({
    'feature': X.columns,
    'rank': rfe.ranking_
}).sort_values('rank')
rfe_ranking


## 6. Method 4: Lasso Regularization


In [ ]:
lasso = Lasso(alpha=0.01, random_state=42)
lasso.fit(X_train_scaled, y_train)

lasso_coef = pd.DataFrame({
    'feature': X.columns,
    'coefficient': lasso.coef_
}).sort_values('coefficient', key=abs, ascending=False)
lasso_coef


In [ ]:
# Features with non-zero coefficients
selected_lasso = lasso_coef[lasso_coef['coefficient'] != 0]
print(f'Lasso retained {len(selected_lasso)} features with non-zero coefficients')
selected_lasso


## 7. Method 5: Mutual Information


In [ ]:
mi_scores = mutual_info_regression(X_train_scaled, y_train, random_state=42)

mi_df = pd.DataFrame({
    'feature': X.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=mi_df, x='mi_score', y='feature')
plt.title('Mutual Information Scores')
plt.tight_layout()
plt.show()
mi_df


## 8. Compare Model Performance with Selected Features


In [ ]:
def evaluate_subset(features, name):
    X_sub = X[features]
    X_tr, X_te, y_tr, y_te = train_test_split(X_sub, y, test_size=0.3, random_state=42)
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_te_sc = scaler.transform(X_te)
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    scores = cross_val_score(rf, X_tr_sc, y_tr, cv=5, scoring='r2')
    return {
        'Method': name,
        'Features': len(features),
        'CV R² Mean': scores.mean().round(4),
        'CV R² Std': scores.std().round(4)
    }

results = []
results.append(evaluate_subset(X.columns.tolist(), 'All Features'))
results.append(evaluate_subset(importance.head(5)['feature'].tolist(), 'RF Importance Top 5'))
results.append(evaluate_subset(selected_kbest.tolist(), 'SelectKBest Top 5'))
results.append(evaluate_subset(selected_rfe.tolist(), 'RFE Top 5'))

pd.DataFrame(results).sort_values('CV R² Mean', ascending=False)


## Summary


In [ ]:
print('''
Feature Selection Methods Summary:

1. Random Forest Importance: built-in, fast, captures non-linear relationships
2. SelectKBest (F-test): fast univariate filter, linear relationships only
3. RFE: iterative, model-aware, computationally expensive
4. Lasso: embedded method, drives irrelevant coefficients to zero
5. Mutual Information: captures any relationship (linear + non-linear)

Best practice: Use multiple methods and take the intersection,
or use domain knowledge to guide final selection.
''')
